In [0]:
from pyspark.sql.types import StructType

TABLE = "cat_adblh_dev.bronze.shipments_postgres_raw"

LANDING = (
    "abfss://landing@stadblhdeveus2001.dfs.core.windows.net/"
    "postgres/shipments"
)

CHECKPOINT_ROOT = (
    "abfss://checkpoints@stadblhdeveus2001.dfs.core.windows.net/"
)
CHECKPOINT_NAME = "q2_shipments_directory_listing_v1"
CHECKPOINT = CHECKPOINT_ROOT + CHECKPOINT_NAME

SOURCE_COLUMNS = [
    "shipment_id",
    "order_id",
    "shipment_status",
    "updated_at",
    "source_version",
]

# Preserve the actual source-field types before clearing Bronze.
existing = spark.table(TABLE)
SOURCE_SCHEMA = StructType([
    existing.schema[name] for name in SOURCE_COLUMNS
])

# Preserve the project's actual source-system label.
labels = [
    row["_source_system"]
    for row in existing.select("_source_system").distinct().collect()
]
if len(labels) != 1 or labels[0] is None:
    raise RuntimeError(
        "Expected one non-null source-system label before the reset."
    )

SOURCE_SYSTEM = labels[0]

# PROBLEM — Clearing Bronze while retaining completed progress
# could prevent already-processed files from refilling the table.
# Require an unused checkpoint for this deliberate rebuild.
checkpoint_entries = dbutils.fs.ls(CHECKPOINT_ROOT)

if any(
    item.name.rstrip("/") == CHECKPOINT_NAME
    for item in checkpoint_entries
):
    raise RuntimeError(
        "This checkpoint already exists. Stop before clearing anything."
    )

print("Source schema and source-system label captured.")
print(f"Checkpoint selected: {CHECKPOINT}")
print("No data has been changed.")

In [0]:
# Run once, after removing the five landed file copies.
# This clears table records, not the table definition or source files.
spark.sql(f"TRUNCATE TABLE {TABLE}")

print("Bronze records cleared. Upload the baseline, then run cell 3.")

In [0]:
from uuid import uuid4
from pyspark.sql import functions as F

# Each execution has its own identifier.
run_id = str(uuid4())

# PROBLEM — Manual delivery selection and last-filename tracking.
# Auto Loader reads the landing root and maintains file-processing state.
source_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.useManagedFileEvents", "false")
    .option("cloudFiles.useNotifications", "false")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.partitionColumns", "")
    .schema(SOURCE_SCHEMA)
    .load(LANDING)
)

# Preserve source fields and record where each row came from.
bronze_stream = (
    source_stream.select(
        *SOURCE_COLUMNS,
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn(
        "_delivery_id",
        F.regexp_extract(
            F.col("_source_file"),
            r"/delivery_id=([^/]+)/",
            1,
        ),
    )
    .withColumn("_run_id", F.lit(run_id))
    .withColumn("_ingested_at", F.current_timestamp())
)

# PROBLEM — Duplicate ingestion during normal restart.
# Reuse this checkpoint and target for every arrival and retry.
query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE)
)

# Wait until this execution finishes before reading the result.
query.awaitTermination()

print(f"Current Bronze rows: {spark.table(TABLE).count()}")

display(
    spark.table(TABLE)
    .groupBy("_delivery_id")
    .agg(F.count("*").alias("rows"))
    .orderBy("_delivery_id")
)